importing libraries chargin data


In [1]:
import json
from joblib import load, dump
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
data = pd.read_csv("data/tickets_inputs_eng_1.csv")
data.head(2)

,complaint_what_happened,ticket_classification,processed_text,relevant_topics
0,I 'm a victim of a bank fraud scam. I applied ...,Checking or savings account + Checking account,bank fraud scam loan pretend inform begin che...,Bank Account Services
1,"1 ) Refinanced home mortgage XXXX, XXXX. Close...",Mortgage + Conventional fixed mortgage,refinanc home mortgag origin loan impound loa...,Mortgage/Loan


exploring balance ratio and basic statistics of the dataset

In [10]:
data.describe()

,complaint_what_happened,ticket_classification,processed_text,relevant_topics
count,15169,15169,15169,15169
unique,15067,76,14991,3
top,Chase has violated 15 USC 1692 by continuing c...,Credit card or prepaid card + General-purpose ...,file disput regard item credit report day have...,Bank Account Services
freq,10,3949,11,5648


In [11]:
data["relevant_topics"].value_counts()

relevant_topics
Bank Account Services            5648
Credit Report or Prepaid Card    5075
Mortgage/Loan                    4446
Name: count, dtype: int64

spliting and  mapping labels before train test split

In [2]:
X_text = data['processed_text']
y_encoded = data['relevant_topics']
def read_idx2label(json_path: str) -> pd.Series:
    """This function read the json file and return a dictionary
    Args:
      json_path (str): path to the json file
     Returns:
      idx2label (dict): dictionary with the mapping"""
    with open(json_path) as f:
        idx2label = json.load(f)
    return idx2label
def decode_labels_into_idx(labels: pd.Series, idx2label: dict) -> pd.Series:
    """This function decode the labels into idx
    Args:
      labels (pd.Series): series with the labels
      idx2label (dict): dictionary with the mapping
     Returns:
      labels (pd.Series): series with the labels decoded
    """
    return labels.map(idx2label)
idx2label = read_idx2label(json_path="data/topic_mapping_1.json")
label2idx = {value: key for key, value in idx2label.items()}


decoding of labels,
vectorization of text,
traintest split,
wich one is the Majority class?



In [3]:
y = decode_labels_into_idx(labels=y_encoded, idx2label=label2idx)
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(X_text)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
majority_class = y_train.value_counts().idxmax()

importing, instancieting and fitting dummy classifier

In [4]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report
dummy_classifier = DummyClassifier(strategy='stratified', constant=majority_class)
dummy_classifier.fit(X_train, y_train)

,strategy,'stratified'
,random_state,None
,constant,'0'


clasification repor of dummy

In [5]:
baseline_predictions = dummy_classifier.predict(X_test)

# Obtener métricas de evaluación del modelo baseline
print(classification_report(y_test, baseline_predictions))

              precision    recall  f1-score   support

           0       0.39      0.36      0.38      1161
           1       0.35      0.36      0.35       999
           2       0.27      0.28      0.28       874

    accuracy                           0.34      3034
   macro avg       0.33      0.33      0.33      3034
weighted avg       0.34      0.34      0.34      3034



vectorizacion de datos importe de librerias instanciacion y entrenamiento de svm Model

In [6]:
from sklearn.svm import SVC
tfidf_vectorizer = TfidfVectorizer()

X_train, X_test, y_train, y_test = train_test_split(X_text, y, test_size=0.2, random_state=42)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# Inicializar y entrenar el modelo SVM
svm_classifier = SVC(kernel='linear', class_weight='balanced', random_state=42)
svm_classifier.fit(X_train_tfidf, y_train)



,C,1.0
,kernel,'linear'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,'balanced'
,verbose,False


predictions and metrics of Suport Vector machine

In [9]:
from sklearn.metrics import accuracy_score
predictions_test = svm_classifier.predict(X_test_tfidf)
predictions_train = svm_classifier.predict(X_train_tfidf)

# Medir la precisión del modelo
accuracy = accuracy_score(y_test, predictions_test)
print(f"Precisión del modelo: {accuracy:.2f}")

# Ver el reporte de clasificación
print(classification_report(y_test, predictions_test))

Precisión del modelo: 0.96
              precision    recall  f1-score   support

           0       0.97      0.96      0.97      1161
           1       0.96      0.97      0.96       999
           2       0.96      0.96      0.96       874

    accuracy                           0.96      3034
   macro avg       0.96      0.96      0.96      3034
weighted avg       0.96      0.96      0.96      3034



In [10]:
unique_classes, counts = np.unique(predictions_test, return_counts=True)

for label, count in zip(unique_classes, counts):
    print(f'Clase: {label}, Cantidad de predicciones: {count}')

Clase: 0, Cantidad de predicciones: 1156
Clase: 1, Cantidad de predicciones: 1008
Clase: 2, Cantidad de predicciones: 870


dump and save svm model

In [12]:
with open ('./model/svm.bin', 'wb') as f_out:
    dump(svm_classifier, f_out)

train_data = pd.DataFrame({
    'texto': X_train,
    'label': y_train,
    'features': X_train_tfidf,
    'predictions': predictions_train
})


# Generar pesos aleatorios para la suma ponderada (número de columnas en la matriz dispersa)
num_features = train_data['features'].iloc[0].shape[1]
weights = np.random.rand(num_features)

# Calcular la suma ponderada para cada fila en 'features' y agregarla como una nueva columna
weighted_sum = train_data['features'].apply(lambda x: np.sum(x.multiply(weights)))
train_data['weighted_sum'] = weighted_sum

test_data = pd.DataFrame({
    'texto': X_test,
    'label': y_test,
    'features': X_test_tfidf,
    'predictions': predictions_test
})

num_features = test_data['features'].iloc[0].shape[1]#words 

weights = np.random.rand(num_features)

# Calcular la suma ponderada para cada fila en 'features' y agregarla como una nueva columna
weighted_sum = test_data['features'].apply(lambda x: np.sum(x.multiply(weights)))
test_data['weighted_sum'] = weighted_sum
#at the end there is a scalar value representing the importance of all words ina  document

Imagine you have a **sparse matrix**.  
In **TF-IDF vectorization**, each document becomes a row vector with the **frequency of the words** it contains from the corpus — pretty sparse.

### A weighted representation is:
Instead of raw counts (1, 2, 3), we assign a **weight** that depends on the **importance of each feature**, representing how **unique** that word is across all documents, or how **important** its semantic meaning is within the document.

### Advantages
You can focus on more important and unique features, allowing for techniques like **PCA** and **clustering** to work more effectively.

### Techniques
- **TF-IDF**: Measures how unique a word is within its document, combined with how frequently it appears across all other documents.  
- **BM25**: Used in recommendation systems or information retrieval. It adds diminishing returns to TF-IDF, balancing document length with word repetition.  
- **Variance-based weighting**: Measures how much each feature (word) varies between samples. It scales down features with low variance (mostly noise), giving equal opportunity to those with higher variance.  
- **Entropy-based weighting**: If the distribution of a feature is more uniform (more random), a higher weight is given.  
- **TF-IDF before cosine similarity**: Helps cluster documents that share rare features in common.  
- **TF-IDF + PCA**: Highlights important features with TF-IDF, and then applies PCA on those weighted features.


Data Quality

In [15]:
from evidently import ColumnMapping
from evidently.report import Report
from evidently.metrics import ColumnDriftMetric, DatasetDriftMetric, DatasetMissingValuesMetric

test_data.head()


ModuleNotFoundError: No module named 'evidently'

In [ ]:
test_data["data_label"]=test_data["data_label"].astype(int
train_data["data_label"]=train_data["data_label"].astype(int)
test_data["predictions"] = test_data["predictions"].astype(int)
train_data["predictions"] = train_data["predictions"].astype(int)
train_data = train_data[["label", "predictions", "weighted_sum"]]
test_data = test_data[["label", "predictions", "weighted_sum"]]

In [3]:
test_data['features_dense'] = test_data['features'].apply(lambda x: x.toarray())
train_data['features_dense'] = train_data['features'].apply(lambda x: x.toarray())
train_data.info()

NameError: name 'test_data' is not defined

concept drift: relationship x y  changes
label drift: distribution of target variable chages
feature drift: covariance in features changes 
signs of this happenning is   metrics  getting worse with no apparent reazon, new categgories in categorical variables, more missing data, feature distributions an covariance  shift. 

it is detected using statistical tests and distance metrics

In [ ]:
colummaping=ColumnMapping(
    target=None,
    prediction='predictions',
    numerical_features='weighted_sum',
    categorical_features=None
    
    
)
report=Report(metrics=
              [ColumnDriftMetric(colum_name='predictions'),
               DatasetDriftMetric(),
               DatasetDriftMetric()])

report.run(reference_data= train_data, current_data= test_data, column_mapping= column_mapping)
report.show(mode='inline')



In [ ]:
data_drift_column_report = Report(metrics=[
    ColumnDriftMetric('weighted_sum'),
    ColumnDriftMetric('weighted_sum', stattest='psi'),
])
data_drift_column_report.run(reference_data=train_data, current_data=test_data)

data_drift_column_report
result = report.as_dict()